# ***TimeBlocks Walkthrough***

## ***Setup***

### ***Imports***

In [1]:
import sys
import tempfile
from pathlib import Path

import numpy as np
import torch

PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "main.py").exists())
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.classes.blocks import ModelRecord
from src.data.classes.enums import Architecture, Task
from src.data.classes.settings import BlockSettings, ClusteringSettings, RouterSettings, SelectionSettings
from src.models.timeblocks.router import Router
from src.services.block_selection import BlockSelector
from src.services.blockbase_store import BlockbaseStore, block_records_of
from src.services.clustering import BlockClusterer
from src.services.coresets import ReservoirBuffer, StreamCore
from src.services.model_building import build_fresh_blocks, build_task_model
from src.services.router_training import fingerprint_matrix
from src.utils.metrics import adjust_predictions, detection_scores
from src.utils.runtime import module_size_megabytes, set_seed

### ***Configuration***

In [2]:
SEED = 2026
BLOCK_SETTINGS = BlockSettings(common_length=128, patch_length=8, hidden_size=16, layers_per_block=3, attention_heads=2)
NUM_BLOCKS = 4
CONTEXT_LENGTHS = (64, 256)
HORIZON = 24
BATCH_SIZE = 8
STREAM_SIZE = 2000
CORESET_CAPACITY = 100
set_seed(seed=SEED)

## ***Blocks***

### ***Multi-Scale Blocks***

In [3]:
def describe_blocks(*, architecture: Architecture) -> list[tuple[int, int, int]]:
    blocks = build_fresh_blocks(
        settings=BLOCK_SETTINGS,
        architecture=architecture,
        num_blocks=NUM_BLOCKS,
    )
    residual = torch.randn(BATCH_SIZE, BLOCK_SETTINGS.common_length)
    rows = []
    for block in blocks:
        output = block(residual=residual)
        rows.append((block.sampling_stride, block.num_tokens, output.shape[-1]))
        residual = output
    return rows

In [4]:
for architecture in Architecture:
    print(architecture.value, describe_blocks(architecture=architecture))

mlp [(8, 2, 128), (4, 4, 128), (2, 8, 128), (1, 16, 128)]
lstm [(8, 2, 128), (4, 4, 128), (2, 8, 128), (1, 16, 128)]
attention [(8, 2, 128), (4, 4, 128), (2, 8, 128), (1, 16, 128)]


### ***Task Models***

In [5]:
forecaster = build_task_model(
    task=Task.FORECASTING,
    settings=BLOCK_SETTINGS,
    blocks=build_fresh_blocks(
        settings=BLOCK_SETTINGS,
        architecture=Architecture.MLP,
        num_blocks=NUM_BLOCKS,
    ),
    context_length=CONTEXT_LENGTHS[0],
    horizon=HORIZON,
    num_classes=1,
)
prediction = forecaster.network(x=torch.randn(BATCH_SIZE, CONTEXT_LENGTHS[0], 1))
print(tuple(prediction.shape), f"{module_size_megabytes(module=forecaster):.3f} MB")

(8, 24, 1) 0.248 MB


## ***Blockbase Routing***

### ***Demo Blockbase***

In [6]:
def build_demo_blockbase(*, root: Path) -> tuple[BlockbaseStore, list[ModelRecord]]:
    store = BlockbaseStore(root=root)
    records = []
    for context_length in CONTEXT_LENGTHS:
        for architecture in Architecture:
            model = build_task_model(
                task=Task.FORECASTING,
                settings=BLOCK_SETTINGS,
                blocks=build_fresh_blocks(
                    settings=BLOCK_SETTINGS,
                    architecture=architecture,
                    num_blocks=NUM_BLOCKS,
                ),
                context_length=context_length,
                horizon=HORIZON,
                num_classes=1,
            )
            record = ModelRecord(
                model_id=f"demo-{architecture.value}-c{context_length}",
                task=Task.FORECASTING.value,
                dataset="synthetic",
                architecture=architecture.value,
                context_length=context_length,
                horizon=HORIZON,
                num_blocks=NUM_BLOCKS,
                output_size=1,
                validation_loss=float("nan"),
                num_parameters=0,
            )
            store.save(
                record=record,
                model=model.network,
                settings=BLOCK_SETTINGS,
            )
            records.append(record)
    return store, records

In [7]:
workspace = tempfile.TemporaryDirectory()
store, records = build_demo_blockbase(root=Path(workspace.name))
fingerprints, blocks = fingerprint_matrix(
    store=store,
    records=records,
)
print(fingerprints.shape, len(blocks))

(24, 128) 24


### ***Clustering***

In [8]:
clusters = BlockClusterer(
    settings=ClusteringSettings(),
    seed=SEED,
).cluster(
    fingerprints=fingerprints,
    context_lengths=[block.context_length for block in blocks],
)
print(clusters.num_clusters, clusters.context_lengths)

2 (64, 256)


### ***Block Selection***

In [9]:
def sample_windows(*, context_length: int) -> torch.Tensor:
    time = torch.arange(context_length, dtype=torch.float32)
    phases = torch.rand(32, 1) * 6.28
    return torch.sin(time / 8 + phases).unsqueeze(-1)

In [10]:
selector = BlockSelector(
    router=Router(
        settings=RouterSettings(),
        fingerprint_size=fingerprints.shape[1],
    ),
    store=store,
    fingerprints=fingerprints,
    blocks=blocks,
    clusters=clusters,
    block_settings=BLOCK_SETTINGS,
    settings=SelectionSettings(num_blocks=3, restrict_to_cluster=False),
    device=torch.device("cpu"),
)
selection, chosen = selector.select(sampler=sample_windows)
print(selection.context_length, selection.block_ids)
print([round(distance, 4) for distance in selection.block_distances])

🔍 Select ▕          ▏   0% • 0/3 • ?block/s • ⏳ 00:00<?

🔍 Select ▕███▎      ▏  33% • 1/3 • 264.52block/s • ⏳ 00:00<00:00, demo-lstm-c64/b2

🔍 Select ▕██████▋   ▏  67% • 2/3 • 127.63block/s • ⏳ 00:00<00:00, demo-attention-c256/b4

🔍 Select ▕██████████▏ 100% • 3/3 • 129.54block/s • ⏳ 00:00<00:00, demo-lstm-c256/b1     

64 ('demo-lstm-c64/b2', 'demo-attention-c256/b4', 'demo-lstm-c256/b1')
[0.786, 0.7706, 0.8163]


## ***StreamCore***

### ***Coreset Spread***

In [11]:
def nearest_neighbour_spread(*, points: np.ndarray) -> float:
    distances = np.linalg.norm(points[:, None, :] - points[None, :, :], axis=-1)
    np.fill_diagonal(distances, np.inf)
    return float(distances.min(axis=1).mean())


def coreset_points(*, coreset, stream: torch.Tensor) -> np.ndarray:
    coreset.insert(
        features=stream,
        indices=torch.arange(len(stream)),
    )
    return stream[coreset.selected()].numpy()

In [12]:
centres = torch.tensor([[0.0, 0.0], [6.0, 6.0]])
stream = centres[torch.randint(0, 2, (STREAM_SIZE,))] + torch.randn(STREAM_SIZE, 2) * torch.where(
    torch.rand(STREAM_SIZE, 1) < 0.95, 0.3, 3.0
)
streamcore = coreset_points(
    coreset=StreamCore(capacity=CORESET_CAPACITY, generator=np.random.default_rng(SEED), device=torch.device("cpu")),
    stream=stream,
)
reservoir = coreset_points(
    coreset=ReservoirBuffer(capacity=CORESET_CAPACITY, generator=np.random.default_rng(SEED)),
    stream=stream,
)
print(f"StreamCore spread {nearest_neighbour_spread(points=streamcore):.3f}")
print(f"Reservoir spread {nearest_neighbour_spread(points=reservoir):.3f}")

StreamCore spread 0.849
Reservoir spread 0.109


## ***Metrics***

### ***Point Adjustment***

In [13]:
labels = np.array([0, 1, 1, 1, 0, 0, 1, 1, 0])
predictions = np.array([0, 0, 1, 0, 0, 0, 0, 0, 1])
print(adjust_predictions(predictions=predictions, labels=labels).astype(int))
print(detection_scores(predictions=predictions, labels=labels, point_adjustment=True))
print(detection_scores(predictions=predictions, labels=labels, point_adjustment=False))
workspace.cleanup()

[0 1 1 1 0 0 0 0 1]
(0.6666666666666665, 0.75, 0.6)
(0.28571428571428575, 0.5, 0.2)
